# 01 · Limpieza y exploración

Entrada: `data/autos_raw.csv` (17.095 avisos, corrida del 2026-10-28).
Salida: `data/autos_clean.csv` y `data/limpieza_reglas.csv` (cuántas filas saca cada regla).

Decisiones (ver `CONTEXTO.md` de la materia, sección 7):

1. Precios en pesos → dólares con la cotización BCU del día del scraping.
2. Se sacan los **0 km**: `km ≤ 1` y año ≥ 2025.
3. `km ≤ 1` con año anterior a 2025 no es 0 km: es **km sin informar** y se saca.
4. Más de **1.000.000 km** = km de relleno (9.999.999, 5.555.555, 1.234.567…), se saca.
5. Precio mayor a **USD 150.000** = outlier, se saca. Sin piso por ahora.
6. **Gemelos** (misma marca, modelo, año, km, precio y moneda) comparten `grupo_gemelo`:
   el split train/test y el K-fold se hacen agrupando por esa columna (notebook de modelos).

La agrupación de modelos raros **no** se hace acá: depende de los conteos de train,
así que va después del split.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data")
FIGS = Path("figuras")
FIGS.mkdir(exist_ok=True)

# Paleta (categórica en orden fijo) y estilo sobrio para el informe
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
TINTA, TINTA_2, GRILLA = "#1f1f1e", "#5f5e58", "#e4e3dd"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA, "text.color": TINTA,
    "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6,
    "lines.linewidth": 2, "font.size": 10,
})

raw = pd.read_csv(DATA / "autos_raw.csv")
raw.shape

## 1. Chequeos de entrada

In [ ]:
assert raw["item_id"].is_unique, "item_id repetido: el scraper ya deduplicaba"
claves = ["marca", "modelo", "anio", "km", "precio", "moneda"]
print("Nulos en columnas clave:")
print(raw[claves].isna().sum())
print()
print(raw["moneda"].value_counts())

## 2. Gemelos

Un gemelo es el mismo auto publicado más de una vez (mismas seis columnas clave).
El grupo se calcula **sobre los datos crudos**, antes de sacar filas, y viaja con cada aviso
hasta el notebook de modelos.

In [ ]:
df = raw.copy()
df["grupo_gemelo"] = df.groupby(claves, sort=False, dropna=False).ngroup()  # dropna=False: 156 avisos sin modelo
tam = df.groupby("grupo_gemelo")["item_id"].transform("size")
print(f"Filas con al menos un gemelo: {(tam > 1).sum()}")
print(f"Grupos con más de una fila:   {df.loc[tam > 1, 'grupo_gemelo'].nunique()}")
print(f"Tamaño máximo de grupo:       {tam.max()}")
assert df["grupo_gemelo"].ge(0).all(), "hay avisos sin grupo"

## 3. Moneda

Cotización BCU (dólar billete, interbancario) del **28/10/2026**, fecha del scraping.
Fuente: bcu.gub.uy → Cotizaciones. **Por ahora se usa un valor provisorio** (ver comentario en la celda).

In [ ]:
# PROVISORIA: 40,255 es un cierre BCU publicado por Ámbito, sin fecha confirmada.
# Reemplazar por el dato oficial del 28/10/2026 (bcu.gub.uy) antes del informe y volver a correr.
COTIZACION_BCU_2026_10_28 = 40.255  # pesos por dólar

if COTIZACION_BCU_2026_10_28 is None:
    raise ValueError("Falta la cotización BCU del 28/10/2026 (celda de arriba).")

df["precio_usd"] = np.where(
    df["moneda"] == "UYU", df["precio"] / COTIZACION_BCU_2026_10_28, df["precio"]
)
print(df.loc[df["moneda"] == "UYU", "precio_usd"].describe().round(0))

## 4. Reglas de limpieza

Cada regla se aplica en orden y se registra cuántas filas saca. Ninguna regla se superpone
con otra sin quedar contada: una fila la saca la primera regla que la toma.

In [ ]:
reglas = [
    ("0 km (km ≤ 1 y año ≥ 2025)",
     lambda d: (d["km"] <= 1) & (d["anio"] >= 2025)),
    ("km sin informar (km ≤ 1 y año < 2025)",
     lambda d: (d["km"] <= 1) & (d["anio"] < 2025)),
    ("km de relleno (km > 1.000.000)",
     lambda d: d["km"] > 1_000_000),
    ("precio > USD 150.000",
     lambda d: d["precio_usd"] > 150_000),
]

registro = [{"regla": "inicio", "sacadas": 0, "quedan": len(df)}]
limpio = df
for nombre, condicion in reglas:
    sacar = condicion(limpio)
    limpio = limpio.loc[~sacar]
    registro.append({"regla": nombre, "sacadas": int(sacar.sum()), "quedan": len(limpio)})

registro = pd.DataFrame(registro)
registro

### Control: km altos que quedan

Por encima de 1.000.000 se sacó como relleno. Lo que queda entre 300.000 y 1.000.000
es raro pero posible (taxis, utilitarios) y se mantiene.

In [ ]:
for umbral in [300_000, 500_000, 900_000]:
    print(f"km > {umbral:>9,}: {(limpio['km'] > umbral).sum():>4} avisos")
limpio.nlargest(8, "km")[["titulo", "anio", "km", "precio_usd"]]

## 5. Variables derivadas

In [ ]:
ANIO_REF = pd.to_datetime(limpio["fecha_scraping"]).dt.year.max()  # 2026
limpio = limpio.assign(
    antiguedad=ANIO_REF - limpio["anio"],          # 2027 → -1 (año modelo adelantado)
    log_precio=np.log(limpio["precio_usd"]),
    log_km=np.log(limpio["km"]),
    tipo_vendedor=limpio["tipo_vendedor"].fillna("desconocido"),
    modelo=limpio["modelo"].fillna("sin_modelo"),  # en modelos van a "otro de la marca"
)
limpio[["antiguedad", "km", "precio_usd", "log_precio", "log_km"]].describe().round(2)

## 6. Exploración

### 6.1 ¿Por qué log(precio)?

El precio tiene cola larga a la derecha: unos pocos autos caros dominan el error cuadrático.
En log la distribución es casi simétrica, y un coeficiente se lee como **variación porcentual**.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(limpio["precio_usd"] / 1000, bins=80, color=SERIES[0], edgecolor="white", linewidth=0.4)
ax[0].set(xlabel="Precio (miles de USD)", ylabel="Avisos", title="Precio")
ax[1].hist(limpio["log_precio"], bins=80, color=SERIES[0], edgecolor="white", linewidth=0.4)
ax[1].set(xlabel="log(precio en USD)", title="log(precio)")
for a in ax: a.grid(axis="x", visible=False)
fig.savefig(FIGS / "01_precio_vs_log.png")
plt.show()
print(f"Asimetría precio: {limpio['precio_usd'].skew():.2f} · log(precio): {limpio['log_precio'].skew():.2f}")

### 6.2 Antigüedad y kilómetros

Si la relación con log(precio) es aproximadamente lineal, el modelo de la propuesta
(`log(precio) = β₀ + β₁·antigüedad + β₂·log(km) + …`) tiene sentido.

In [ ]:
med = limpio.groupby("antiguedad")["log_precio"].agg(["median", "size"])
med = med[med["size"] >= 30]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(limpio["antiguedad"], limpio["log_precio"], s=4, alpha=0.08, color=TINTA_2, linewidths=0)
ax[0].plot(med.index, med["median"], color=SERIES[0], marker="o", markersize=4, label="mediana")
ax[0].set(xlabel="Antigüedad (años)", ylabel="log(precio en USD)", title="Antigüedad")
ax[0].legend(frameon=False)
hb = ax[1].hexbin(limpio["log_km"], limpio["log_precio"], gridsize=45, cmap="Blues", mincnt=1, bins="log")
ax[1].set(xlabel="log(km)", title="Kilómetros")
fig.colorbar(hb, ax=ax[1], label="avisos (escala log)")
fig.savefig(FIGS / "02_antiguedad_km.png")
plt.show()
print(limpio[["log_precio", "antiguedad", "log_km"]].corr().round(2))

### 6.3 ¿La depreciación es igual en todas las marcas? (pregunta secundaria)

Mediana del precio por antigüedad para las 6 marcas con más avisos, en escala log:
**pendientes paralelas** = misma depreciación porcentual; pendientes distintas = la marca importa.

In [ ]:
top = limpio["marca"].value_counts().head(6).index
fig, ax = plt.subplots(figsize=(8, 4.2))
for color, marca in zip(SERIES, top):
    m = (limpio[limpio["marca"] == marca]
         .groupby("antiguedad")["precio_usd"].agg(["median", "size"]))
    m = m[(m["size"] >= 15) & (m.index >= 0) & (m.index <= 20)]
    ax.plot(m.index, m["median"], color=color, label=marca)
ax.set_yscale("log")
ticks = [6000, 8000, 10000, 15000, 20000, 30000]
ax.yaxis.set_major_locator(plt.FixedLocator(ticks))
ax.yaxis.set_minor_locator(plt.NullLocator())
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f} k"))
ax.set_xticks(range(0, 21, 2))
ax.set(xlabel="Antigüedad (años)", ylabel="Mediana del precio (USD, escala log)",
       title="Depreciación por marca (6 marcas con más avisos)")
ax.legend(frameon=False, ncol=3, fontsize=8)
fig.savefig(FIGS / "03_depreciacion_marca.png")
plt.show()

# Pendiente por marca: log(precio) ~ antigüedad, ajuste simple (exploratorio)
filas = []
for marca in top:
    d = limpio[(limpio["marca"] == marca) & limpio["antiguedad"].between(0, 20)]
    b1 = np.polyfit(d["antiguedad"], d["log_precio"], 1)[0]
    filas.append({"marca": marca, "avisos": len(d), "pierde_por_año_%": round((1 - np.exp(b1)) * 100, 1)})
pd.DataFrame(filas)

### 6.4 Concesionaria vs particular, y modelos raros

In [ ]:
print(limpio.groupby("tipo_vendedor")["precio_usd"].agg(["size", "median"]).round(0))
print()
vc = limpio["modelo"].value_counts()
for u in [5, 10, 20, 50]:
    print(f"Modelos con < {u:>2} avisos: {(vc < u).sum():>3} modelos, {vc[vc < u].sum():>5} avisos")

## 7. Guardar

`grupo_gemelo` queda en el CSV: el notebook de modelos lo usa para `GroupShuffleSplit` y `GroupKFold`.

In [ ]:
cols = ["item_id", "grupo_gemelo", "marca", "modelo", "version", "anio", "antiguedad",
        "km", "log_km", "precio", "moneda", "precio_usd", "log_precio",
        "tipo_vendedor", "tienda_oficial", "departamento", "barrio",
        "transmision_titulo", "combustible_titulo", "motor_titulo"]
limpio[cols].to_csv(DATA / "autos_clean.csv", index=False)
registro.to_csv(DATA / "limpieza_reglas.csv", index=False)
print(f"Guardado: {len(limpio)} avisos, {limpio['grupo_gemelo'].nunique()} grupos de gemelos")